<a href="https://colab.research.google.com/github/marcelalozano27-ship-it/Assignment4_Fong_Marcela/blob/main/Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NER Notebook 1: Inference & Rule-Based NER
## Social Media Domain — Broad Twitter Corpus

**BSAN 6200 — Text Mining & Social Media Analytics**

## What You'll Learn
1. **Load the Broad Twitter Corpus** from HuggingFace
2. **Pretrained NER inference** using spaCy, NLTK, and HuggingFace Transformers
3. **Error analysis** — False Positives, False Negatives, wrong labels
4. **Rule-based NER** using regex patterns and spaCy EntityRuler for tweet-specific entities
5. **Export** predictions to CoNLL format and Label Studio JSON for annotation

## Dataset
- **Name:** Broad Twitter Corpus (GateNLP)
- **Source:** https://huggingface.co/datasets/GateNLP/broad_twitter_corpus
- **Entity Types:** PERSON, LOCATION, ORGANIZATION
- **Domain:** Tweets — short, informal, noisy social media text
- **Why this domain?** Twitter/X text has unique NER challenges: abbreviations, hashtags,
  @mentions, slang, lack of capitalization, and rapidly changing named entities.

---

---
## Part 1: Load the Broad Twitter Corpus

The Broad Twitter Corpus contains ~9,000 annotated tweets.
We will load it from HuggingFace and inspect a sample.

**Key challenges in social media NER:**
- Inconsistent capitalization (`elon musk` vs `Elon Musk`)
- Hashtags and mentions (`#Apple`, `@realDonaldTrump`)
- Abbreviations and slang (`tbh`, `smh`, `lol`)
- Missing punctuation and grammar
- URLs embedded in text


In [ ]:
# ============================================================
# SETUP
# ============================================================

!pip install -q "datasets<4.0.0" spacy transformers seqeval torch
!python -m spacy download en_core_web_sm

from datasets import load_dataset
import spacy
from spacy import displacy
import nltk
from transformers import pipeline
import json
import re
from collections import Counter

# NLTK downloads
nltk.download("punkt", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("maxent_ne_chunker", quiet=True)
nltk.download("words", quiet=True)

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

print(f"spaCy model loaded: {nlp.meta['name']}")
print(f"Pipeline components: {nlp.pipe_names}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 18.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
spaCy model loaded: core_web_sm
Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [ ]:
from datasets import load_dataset

dataset = load_dataset("GateNLP/broad_twitter_corpus")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

broad_twitter_corpus.py: 0.00B [00:00, ?B/s]

broad-twitter-corpus/train/0000.parquet:   0%|          | 0.00/449k [00:00<?, ?B/s]

broad-twitter-corpus/validation/0000.par(…):   0%|          | 0.00/163k [00:00<?, ?B/s]

broad-twitter-corpus/test/0000.parquet:   0%|          | 0.00/192k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5342 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2002 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2002 [00:00<?, ? examples/s]

In [ ]:
# ============================================================
# DECODE TAGS: Map integer tag IDs to string labels
# ============================================================

# Get the tag names from the dataset features
TAG_NAMES = dataset['train'].features['ner_tags'].feature.names
print("Tag mapping:")
for i, name in enumerate(TAG_NAMES):
    print(f"  {i} → {name}")

def decode_tags(token_list, tag_id_list):
    """Convert (tokens, tag_ids) to (token, tag_string) pairs."""
    return [(tok, TAG_NAMES[tid]) for tok, tid in zip(token_list, tag_id_list)]

# Show 5 annotated examples from the training set
print("\n--- Sample Annotated Tweets ---")
for i in range(5):
    ex = dataset['train'][i]
    pairs = decode_tags(ex['tokens'], ex['ner_tags'])
    sentence = " ".join(ex['tokens'])
    entities = [(tok, tag) for tok, tag in pairs if tag != "O"]
    print(f"\nTweet {i+1}: {sentence}")
    if entities:
        for tok, tag in entities:
            print(f"  → {tok:<25} {tag}")
    else:
        print("  → No entities")


Tag mapping:
  0 → O
  1 → B-PER
  2 → I-PER
  3 → B-ORG
  4 → I-ORG
  5 → B-LOC
  6 → I-LOC

--- Sample Annotated Tweets ---

Tweet 1: I hate the words chunder , vomit and puke . BUUH .
  → No entities

Tweet 2: ♥ . . ) ) ( ♫ . ( ړײ ) ♫ . ♥ . « ▓ » ♥ . ♫ . . ╝ ╚ . . ♫ Happy New Year
  → No entities

Tweet 3: Alesan kenapa mlm kita lbh srg galau Poconggg '"' TwitFAKTA : Otak lebih aktif di malam hari dari pada di pagi hari . # TwitFAKTA '"'
  → No entities

Tweet 4: Complete Tosca on the tube http://t.co/O90deSLB
  → No entities

Tweet 5: Think you call that smash and grab . # Gateshead 's media man just admitted to me it was '"' daylight robbery '"' . Shaw 's only touch was his goal .
  → Gateshead                 B-LOC
  → Shaw                      B-PER


In [ ]:
# ============================================================
# DATASET STATISTICS: Understand the corpus
# ============================================================

all_tags = [TAG_NAMES[tid]
            for split in ['train', 'validation', 'test']
            for ex in dataset[split]
            for tid in ex['ner_tags']]

entity_tags = [t for t in all_tags if t != "O"]
entity_types = Counter(t.split("-", 1)[1] for t in entity_tags if "-" in t)

total_tweets = sum(len(dataset[s]) for s in ['train', 'validation', 'test'])

print("BROAD TWITTER CORPUS — STATISTICS")
print("=" * 50)
print(f"Total tweets       : {total_tweets:,}")
print(f"Total tokens       : {len(all_tags):,}")
print(f"Entity tokens      : {len(entity_tags):,} ({len(entity_tags)/len(all_tags)*100:.1f}%)")
print(f"Non-entity tokens  : {len(all_tags)-len(entity_tags):,} ({(len(all_tags)-len(entity_tags))/len(all_tags)*100:.1f}%)")
print(f"\nEntity type distribution:")
for etype, count in entity_types.most_common():
    bar = "█" * (count // 50)
    print(f"  {etype:<15} {count:>5}  {bar}")


BROAD TWITTER CORPUS — STATISTICS
Total tweets       : 9,346
Total tokens       : 150,388
Entity tokens      : 18,573 (12.4%)
Non-entity tokens  : 131,815 (87.6%)

Entity type distribution:
  PER              9482  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  ORG              5311  ██████████████████████████████████████████████████████████████████████████████████████████████████████████
  LOC              3780  ███████████████████████████████████████████████████████████████████████████


---
## Part 2: Pretrained NER with spaCy

We run spaCy's `en_core_web_sm` on the Twitter corpus and compare
its predictions against the gold labels.

spaCy was trained on **web news and Wikipedia** — very different from tweets.
This will help us identify failure modes and motivate custom annotation.


In [ ]:
# ============================================================
# RUN spaCy NER ON TWITTER SAMPLES
# ============================================================
# Reconstruct the raw text from token lists, then run spaCy

# Take 200 tweets from the training set for error analysis
sample_tweets = dataset['train'].select(range(200))

def reconstruct_text(tokens):
    """Join tokens with spaces — simple reconstruction for Twitter."""
    return " ".join(tokens)

# Run spaCy on sample tweets
print("Running spaCy NER on 200 sample tweets...\n")

spacy_results = []
for ex in sample_tweets:
    text = reconstruct_text(ex['tokens'])
    doc = nlp(text)
    gold_pairs = decode_tags(ex['tokens'], ex['ner_tags'])

    spacy_ents = [(ent.text, ent.label_) for ent in doc.ents]
    gold_ents  = [(tok, tag.split("-", 1)[1])
                  for tok, tag in gold_pairs if tag != "O"]

    spacy_results.append({
        "text": text,
        "gold": gold_ents,
        "spacy": spacy_ents,
        "tokens": ex['tokens'],
        "gold_tags": [TAG_NAMES[t] for t in ex['ner_tags']]
    })

print(f"Processed {len(spacy_results)} tweets")

# Show the first 5
print("\n--- Sample Predictions vs Gold ---")
for r in spacy_results[:5]:
    print(f"\nTweet: {r['text']}")
    print(f"  Gold  : {r['gold']}")
    print(f"  spaCy : {r['spacy']}")


Running spaCy NER on 200 sample tweets...

Processed 200 tweets

--- Sample Predictions vs Gold ---

Tweet: I hate the words chunder , vomit and puke . BUUH .
  Gold  : []
  spaCy : []

Tweet: ♥ . . ) ) ( ♫ . ( ړײ ) ♫ . ♥ . « ▓ » ♥ . ♫ . . ╝ ╚ . . ♫ Happy New Year
  Gold  : []
  spaCy : [('♥', 'ORG'), ('╝ ╚', 'PERSON'), ('♫ Happy New Year', 'ORG')]

Tweet: Alesan kenapa mlm kita lbh srg galau Poconggg '"' TwitFAKTA : Otak lebih aktif di malam hari dari pada di pagi hari . # TwitFAKTA '"'
  Gold  : []
  spaCy : [('Alesan', 'ORG'), ('mlm kita', 'PERSON'), ('Poconggg', 'GPE'), ('Otak', 'PERSON'), ('aktif di malam hari dari pada di pagi hari', 'PERSON')]

Tweet: Complete Tosca on the tube http://t.co/O90deSLB
  Gold  : []
  spaCy : [('Complete Tosca', 'PERSON'), ('http://t.co/O90deSLB', 'GPE')]

Tweet: Think you call that smash and grab . # Gateshead 's media man just admitted to me it was '"' daylight robbery '"' . Shaw 's only touch was his goal .
  Gold  : [('Gateshead', 'LOC'), ('Shaw'

In [ ]:
# ============================================================
# VISUALIZE spaCy ON TWEET-SPECIFIC EXAMPLES
# ============================================================
# These examples show common spaCy failure modes on tweets

demo_tweets = [
    "RT @BarackObama: The future belongs to those who believe in their dreams #USA",
    "just saw elon musk tweet about dogecoin again lol $DOGE going crazy",
    "Apple stock down 3% after earnings call. Tim Cook says supply chains recovering",
    "Manchester United lost AGAIN smh… Old Trafford was silent",
    "meeting in SF next week with the Google team 🙌",
]

print("spaCy Visualization on Demo Tweets:")
print("=" * 60)
for tweet in demo_tweets:
    doc = nlp(tweet)
    print(f"\nTweet: {tweet}")
    displacy.render(doc, style="ent", jupyter=True, options={'distance': 120})
    for ent in doc.ents:
        print(f"  → {ent.text:<30} {ent.label_}")
    if not doc.ents:
        print("  → No entities detected!")


spaCy Visualization on Demo Tweets:

Tweet: RT @BarackObama: The future belongs to those who believe in their dreams #USA


/usr/local/lib/python3.12/dist-packages/spacy/displacy/__init__.py:214: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


  → No entities detected!

Tweet: just saw elon musk tweet about dogecoin again lol $DOGE going crazy


  → DOGE                           PERSON

Tweet: Apple stock down 3% after earnings call. Tim Cook says supply chains recovering


  → Apple                          ORG
  → 3%                             PERCENT
  → Tim Cook                       PERSON

Tweet: Manchester United lost AGAIN smh… Old Trafford was silent


  → Manchester United              PERSON
  → AGAIN                          ORG

Tweet: meeting in SF next week with the Google team 🙌


  → SF                             GPE
  → next week                      DATE
  → Google                         ORG


In [ ]:
# ============================================================
# ERROR ANALYSIS: Categorize spaCy failures on Twitter
# ============================================================
# Three types of errors:
#   FP (False Positive / Spurious) — spaCy found something not in gold
#   FN (False Negative / Missed)   — gold entity that spaCy missed
#   WL (Wrong Label)               — found same span but wrong type

# Label mapping: Twitter corpus uses PER/ORG/LOC
# spaCy uses PERSON/ORG/GPE/LOC/NORP/DATE/MONEY/PRODUCT...
LABEL_MAP = {"PER": "PERSON", "LOC": ["GPE", "LOC"], "ORG": "ORG"}

def normalize_spacy_label(label):
    """Map spaCy labels to Broad Twitter Corpus labels where possible."""
    if label == "PERSON":
        return "PER"
    elif label in ("GPE", "LOC", "FAC"):
        return "LOC"
    elif label == "ORG":
        return "ORG"
    return label  # MISC, DATE, etc.

false_positives = []
false_negatives = []
wrong_labels    = []

for r in spacy_results:
    gold_set  = set((t.lower(), tag) for t, tag in r['gold'])
    spacy_set = set((t.lower(), normalize_spacy_label(tag)) for t, tag in r['spacy'])

    gold_texts  = {t.lower() for t, _ in r['gold']}
    spacy_texts = {t.lower() for t, _ in r['spacy']}

    for text, tag in (spacy_set - gold_set):
        if text not in gold_texts:
            false_positives.append({"entity": text, "pred_label": tag,
                                     "tweet": r['text']})
        else:
            gold_label = next((g for t, g in r['gold'] if t.lower() == text), "?")
            wrong_labels.append({"entity": text, "pred": tag,
                                  "gold": gold_label, "tweet": r['text']})

    for text, tag in r['gold']:
        if text.lower() not in spacy_texts:
            false_negatives.append({"entity": text, "gold_label": tag,
                                     "tweet": r['text']})

print("ERROR ANALYSIS SUMMARY (spaCy on 200 Twitter samples)")
print("=" * 60)
print(f"False Positives (Spurious) : {len(false_positives)}")
print(f"False Negatives (Missed)   : {len(false_negatives)}")
print(f"Wrong Labels               : {len(wrong_labels)}")

print(f"\n--- Top 10 Missed Entities (False Negatives) ---")
for ex in false_negatives[:10]:
    print(f"  • '{ex['entity']}' ({ex['gold_label']})")
    print(f"    Tweet: {ex['tweet'][:80]}...")

print(f"\n--- Top 5 Spurious Entities (False Positives) ---")
for ex in false_positives[:5]:
    print(f"  • '{ex['entity']}' predicted as {ex['pred_label']}")
    print(f"    Tweet: {ex['tweet'][:80]}...")

print(f"\n--- Top 5 Wrong Labels ---")
for ex in wrong_labels[:5]:
    print(f"  • '{ex['entity']}': Gold={ex['gold']}, Pred={ex['pred']}")


ERROR ANALYSIS SUMMARY (spaCy on 200 Twitter samples)
False Positives (Spurious) : 199
False Negatives (Missed)   : 91
Wrong Labels               : 17

--- Top 10 Missed Entities (False Negatives) ---
  • 'blink' (ORG)
    Tweet: middle aged man band playing blink 182 . l0 l ....
  • '.' (ORG)
    Tweet: middle aged man band playing blink 182 . l0 l ....
  • 'Dorothy' (PER)
    Tweet: How did Dorothy Gale come back * younger * in Return to Oz ? ?...
  • 'Gale' (PER)
    Tweet: How did Dorothy Gale come back * younger * in Return to Oz ? ?...
  • 'the' (ORG)
    Tweet: TD Chargers = \ . If we ca n't beat the Chargers we do n't deserve to go to the ...
  • 'saffron' (PER)
    Tweet: Random boy I 've never met before said to me last night ' you 're saffron coe , ...
  • 'coe' (PER)
    Tweet: Random boy I 've never met before said to me last night ' you 're saffron coe , ...
  • 'stamford' (LOC)
    Tweet: Random boy I 've never met before said to me last night ' you 're saffron coe , ...

In [ ]:
# ============================================================
# ERROR PATTERN ANALYSIS: Why does spaCy fail on Twitter?
# ============================================================

# Analyze common patterns in missed entities
missed_lower = [e['entity'].lower() for e in false_negatives]
missed_patterns = {
    "has_@"    : sum(1 for e in false_negatives if "@" in e['entity']),
    "has_#"    : sum(1 for e in false_negatives if "#" in e['entity']),
    "all_lower": sum(1 for e in false_negatives if e['entity'].islower()),
    "all_upper": sum(1 for e in false_negatives if e['entity'].isupper()),
    "has_emoji": sum(1 for e in false_negatives if any(ord(c) > 127 for c in e['entity'])),
    "1_token"  : sum(1 for e in false_negatives if len(e['entity'].split()) == 1),
    "2+ tokens": sum(1 for e in false_negatives if len(e['entity'].split()) > 1),
}

print("PATTERNS IN MISSED ENTITIES (spaCy False Negatives)")
print("=" * 50)
for pattern, count in sorted(missed_patterns.items(), key=lambda x: -x[1]):
    pct = count / max(len(false_negatives), 1) * 100
    print(f"  {pattern:<20} {count:>4}  ({pct:.1f}%)")

print(f"\nKey finding: spaCy struggles with lowercase entities and Twitter-specific")
print(f"formats (@mentions, #hashtags). This motivates domain-specific annotation.")


PATTERNS IN MISSED ENTITIES (spaCy False Negatives)
  1_token                91  (100.0%)
  all_lower              26  (28.6%)
  all_upper               7  (7.7%)
  has_@                   0  (0.0%)
  has_#                   0  (0.0%)
  has_emoji               0  (0.0%)
  2+ tokens               0  (0.0%)

Key finding: spaCy struggles with lowercase entities and Twitter-specific
formats (@mentions, #hashtags). This motivates domain-specific annotation.


---
## Part 3: Pretrained NER with NLTK

NLTK uses a multi-step pipeline: tokenize → POS tag → NE chunk.
We compare it against spaCy on the same Twitter samples.

NLTK is older and less accurate but helps us understand the pipeline.


In [ ]:
# ============================================================
# NLTK NER ON TWITTER SAMPLES
# ============================================================

nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('maxent_ne_chunker_tab', quiet=True)

demo_tweets = [
    "Apple just announced the iPhone 16 at their Cupertino event",
    "Elon Musk bought Twitter for $44 billion last year",
    "Barcelona won Champions League final vs Manchester City",
    "Amazon Prime Day starts next Tuesday with massive discounts",
]

print("NLTK NER Pipeline on Twitter Samples")
print("=" * 60)

for tweet in demo_tweets:
    print(f"\nTweet: {tweet}")

    # Step 1: Tokenize
    tokens = nltk.word_tokenize(tweet)

    # Step 2: POS tag
    pos_tags = nltk.pos_tag(tokens)

    # Step 3: Named Entity Chunking
    ne_tree = nltk.ne_chunk(pos_tags)

    # Extract entities from the tree
    entities = []
    for subtree in ne_tree:
        if hasattr(subtree, 'label'):
            entity_text = " ".join(word for word, pos in subtree.leaves())
            entity_type = subtree.label()
            entities.append((entity_text, entity_type))
            print(f"  → {entity_text:<25} {entity_type}")

    if not entities:
        print("  → No entities detected!")

NLTK NER Pipeline on Twitter Samples

Tweet: Apple just announced the iPhone 16 at their Cupertino event
  → Apple                     GPE
  → iPhone                    ORGANIZATION

Tweet: Elon Musk bought Twitter for $44 billion last year
  → Elon                      PERSON
  → Musk                      PERSON
  → Twitter                   PERSON

Tweet: Barcelona won Champions League final vs Manchester City
  → Barcelona                 PERSON
  → Champions League          PERSON
  → Manchester City           PERSON

Tweet: Amazon Prime Day starts next Tuesday with massive discounts
  → Amazon                    PERSON


### spaCy vs NLTK on Twitter

| Aspect | spaCy | NLTK |
|--------|-------|------|
| Handles lowercase tweets | ❌ Often misses | ❌ Often misses |
| Entity types | 18 types | ~3 types |
| @mentions / #hashtags | ❌ Misses | ❌ Misses |
| Speed | Fast | Slow |
| Best use | Production baseline | Teaching the pipeline |

**Key insight:** Both struggle with informal Twitter text. This motivates
training a custom model on Twitter-specific annotations.


---
## Part 4: Pretrained NER with HuggingFace Transformers

We use `dslim/bert-base-NER`, a BERT model fine-tuned on CoNLL-2003.
Transformers generally outperform spaCy and NLTK but still struggle
with Twitter-specific noise.


In [ ]:
# ============================================================
# SETUP: Load HuggingFace NER pipeline
# ============================================================
from transformers import pipeline as hf_pipeline

# dslim/bert-base-NER is fine-tuned on CoNLL-2003 (news domain)
# Entity types: PER, ORG, LOC, MISC
model_name = "dslim/bert-base-NER"

ner_pipeline = hf_pipeline(
    "ner",
    model=model_name,
    aggregation_strategy="simple"
)

print(f"Model loaded: {model_name}")
print("Entity types: PER (person), ORG (organization), LOC (location), MISC (miscellaneous)")


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded: dslim/bert-base-NER
Entity types: PER (person), ORG (organization), LOC (location), MISC (miscellaneous)


In [ ]:
# ============================================================
# RUN TRANSFORMER NER ON TWITTER SAMPLES
# ============================================================

hf_demo_tweets = [
    "Elon Musk is buying Twitter for 44 billion dollars",
    "Apple CEO Tim Cook spoke at a conference in San Francisco",
    "Liverpool beat Manchester United 3-0 at Anfield",
    "just saw aoc speak at the democratic national convention",
    "Tesla stock up 8% after Q3 earnings beat Wall Street expectations",
]

print("HuggingFace BERT-NER on Twitter Samples")
print("=" * 60)

for tweet in hf_demo_tweets:
    print(f"\nTweet: {tweet}")
    results = ner_pipeline(tweet)

    if results:
        for r in results:
            print(f"  → {r['word']:<25} {r['entity_group']:<8} "
                  f"(confidence: {r['score']:.3f})")
    else:
        print("  → No entities detected!")


HuggingFace BERT-NER on Twitter Samples

Tweet: Elon Musk is buying Twitter for 44 billion dollars
  → Elon Musk                 ORG      (confidence: 0.997)
  → Twitter                   ORG      (confidence: 0.981)

Tweet: Apple CEO Tim Cook spoke at a conference in San Francisco
  → Apple                     ORG      (confidence: 0.999)
  → Tim Cook                  PER      (confidence: 1.000)
  → San Francisco             LOC      (confidence: 0.999)

Tweet: Liverpool beat Manchester United 3-0 at Anfield
  → Liverpool                 ORG      (confidence: 1.000)
  → Manchester United         ORG      (confidence: 0.999)
  → Anfield                   LOC      (confidence: 0.998)

Tweet: just saw aoc speak at the democratic national convention
  → No entities detected!

Tweet: Tesla stock up 8% after Q3 earnings beat Wall Street expectations
  → Tesla                     ORG      (confidence: 0.993)
  → Wall Street               LOC      (confidence: 0.998)


In [ ]:
# ============================================================
# VISUALIZE TRANSFORMER RESULTS
# ============================================================

for tweet in hf_demo_tweets[:4]:
    results = ner_pipeline(tweet)
    ents = [{"start": r["start"], "end": r["end"], "label": r["entity_group"]}
            for r in results]
    doc_data = {"text": tweet, "ents": ents, "title": None}
    displacy.render(doc_data, style="ent", manual=True, jupyter=True)


In [ ]:
# ============================================================
# COMPARE ALL THREE MODELS ON THE SAME TWEETS
# ============================================================

comparison_tweets = [
    "Apple CEO Tim Cook announced new iPhone in San Francisco",
    "elon musk tweeted again about bitcoin going to the moon lol",
    "Manchester United fans upset after Old Trafford defeat",
]

print("MODEL COMPARISON: spaCy vs NLTK vs BERT-NER")
print("=" * 70)

for tweet in comparison_tweets:
    print(f"\nTweet: {tweet}")
    print("-" * 60)

    # spaCy
    doc = nlp(tweet)
    spacy_ents = [(e.text, e.label_) for e in doc.ents]
    print(f"  spaCy : {spacy_ents if spacy_ents else 'None'}")

    # NLTK
    tokens = nltk.word_tokenize(tweet)
    ne_tree = nltk.ne_chunk(nltk.pos_tag(tokens))
    nltk_ents = [(" ".join(w for w, _ in st.leaves()), st.label())
                 for st in ne_tree if hasattr(st, 'label')]
    print(f"  NLTK  : {nltk_ents if nltk_ents else 'None'}")

    # HuggingFace
    hf_res = ner_pipeline(tweet)
    hf_ents = [(r['word'], r['entity_group']) for r in hf_res]
    print(f"  BERT  : {hf_ents if hf_ents else 'None'}")


MODEL COMPARISON: spaCy vs NLTK vs BERT-NER

Tweet: Apple CEO Tim Cook announced new iPhone in San Francisco
------------------------------------------------------------
  spaCy : [('Apple', 'ORG'), ('Tim Cook', 'PERSON'), ('iPhone', 'ORG'), ('San Francisco', 'GPE')]
  NLTK  : [('Apple', 'PERSON'), ('CEO Tim Cook', 'ORGANIZATION'), ('San', 'GPE')]
  BERT  : [('Apple', 'ORG'), ('Tim Cook', 'PER'), ('iPhone', 'MISC'), ('San Francisco', 'LOC')]

Tweet: elon musk tweeted again about bitcoin going to the moon lol
------------------------------------------------------------
  spaCy : [('elon musk', 'PERSON')]
  NLTK  : None
  BERT  : None

Tweet: Manchester United fans upset after Old Trafford defeat
------------------------------------------------------------
  spaCy : [('Manchester United', 'PERSON'), ('Old Trafford', 'ORG')]
  NLTK  : [('United', 'GPE'), ('Old Trafford', 'PERSON')]
  BERT  : [('Manchester United', 'ORG'), ('Old Trafford', 'ORG')]


---
## Part 5: Rule-Based NER for Twitter-Specific Entities

Twitter text has structured patterns that regex can capture reliably:
- **@mentions** → likely PERSON or ORG
- **#hashtags** → might contain entity names
- **$cashtags** → stock tickers (PRODUCT/ORG)
- **URLs** → not entities, should be excluded
- **RT @user:** → retweet pattern


In [ ]:
# ============================================================
# REGEX-BASED NER FOR TWITTER PATTERNS
# ============================================================

import re

# Define Twitter-specific regex patterns
PATTERNS = {
    "MENTION"  : re.compile(r'@[A-Za-z0-9_]{1,50}'),
    "HASHTAG"  : re.compile(r'#[A-Za-z][A-Za-z0-9_]{1,49}'),
    "CASHTAG"  : re.compile(r'\$[A-Z]{1,5}\b'),          # $TSLA, $AAPL
    "URL"      : re.compile(r'https?://\S+|www\.\S+'),
    "RT_HEADER": re.compile(r'^RT\s+@[A-Za-z0-9_]+:'),
}

def extract_twitter_entities(text):
    """
    Extract Twitter-specific entities using regex patterns.

    Args:
        text: raw tweet string

    Returns:
        list of (entity_text, start, end, label) tuples
    """
    entities = []
    for label, pattern in PATTERNS.items():
        for match in pattern.finditer(text):
            entities.append((match.group(), match.start(), match.end(), label))
    # Sort by start position
    entities.sort(key=lambda x: x[1])
    return entities

# Test on sample tweets
test_tweets = [
    "RT @ESPN: Breaking: @LeBronJames signs with @Lakers for $97M #NBA",
    "Just bought more $TSLA and $NVDA 📈 check this thread https://t.co/xyz",
    "#AppleEvent 2025: Tim Cook reveals @Apple Vision Pro upgrades",
    "no way @elonmusk really tweeted that lmao 💀 #Twitter",
]

print("Twitter-Specific Rule-Based NER")
print("=" * 60)

for tweet in test_tweets:
    print(f"\nTweet: {tweet}")
    entities = extract_twitter_entities(tweet)
    if entities:
        for text, start, end, label in entities:
            print(f"  → {text:<30} {label}  (chars {start}-{end})")
    else:
        print("  → No entities found")


Twitter-Specific Rule-Based NER

Tweet: RT @ESPN: Breaking: @LeBronJames signs with @Lakers for $97M #NBA
  → RT @ESPN:                      RT_HEADER  (chars 0-9)
  → @ESPN                          MENTION  (chars 3-8)
  → @LeBronJames                   MENTION  (chars 20-32)
  → @Lakers                        MENTION  (chars 44-51)
  → #NBA                           HASHTAG  (chars 61-65)

Tweet: Just bought more $TSLA and $NVDA 📈 check this thread https://t.co/xyz
  → $TSLA                          CASHTAG  (chars 17-22)
  → $NVDA                          CASHTAG  (chars 27-32)
  → https://t.co/xyz               URL  (chars 53-69)

Tweet: #AppleEvent 2025: Tim Cook reveals @Apple Vision Pro upgrades
  → #AppleEvent                    HASHTAG  (chars 0-11)
  → @Apple                         MENTION  (chars 35-41)

Tweet: no way @elonmusk really tweeted that lmao 💀 #Twitter
  → @elonmusk                      MENTION  (chars 7-16)
  → #Twitter                       HASHTAG  (chars 44-5

In [ ]:
# ============================================================
# SPACYENTITYRULER: Add Twitter patterns to spaCy pipeline
# ============================================================

# Create a new spaCy model with both default NER + Twitter rules
nlp_twitter = spacy.load("en_core_web_sm")

# Add EntityRuler BEFORE the default NER — our rules take priority
ruler = nlp_twitter.add_pipe("entity_ruler", before="ner")

# Token-level patterns for Twitter entities
twitter_patterns = [
    # @mention → PERSON (will be refined during annotation)
    {"label": "PERSON",  "pattern": [{"TEXT": {"REGEX": r"@[A-Za-z0-9_]+"}}]},
    # $cashtag → PRODUCT (stock ticker)
    {"label": "PRODUCT", "pattern": [{"TEXT": {"REGEX": r"\$[A-Z]{2,5}"}}]},
    # Common social media company names
    {"label": "ORG",     "pattern": [{"LOWER": "twitter"}]},
    {"label": "ORG",     "pattern": [{"LOWER": "instagram"}]},
    {"label": "ORG",     "pattern": [{"LOWER": "tiktok"}]},
    {"label": "ORG",     "pattern": [{"LOWER": "facebook"}]},
    {"label": "ORG",     "pattern": [{"LOWER": "snapchat"}]},
    # Sports teams (common in Twitter)
    {"label": "ORG",     "pattern": [{"LOWER": "lakers"}]},
    {"label": "ORG",     "pattern": [{"LOWER": "warriors"}]},
    {"label": "ORG",     "pattern": [{"LOWER": "chelsea"}]},
    {"label": "ORG",     "pattern": [{"LOWER": "arsenal"}]},
]

ruler.add_patterns(twitter_patterns)
print(f"Added {len(twitter_patterns)} Twitter-specific patterns")

# Test the augmented pipeline
test_tweets_ruler = [
    "Just followed @BarackObama on Twitter, amazing content",
    "Bought $TSLA at 200, now at 250 thanks to Elon",
    "lakers vs warriors tonight 🏀 who's winning?",
    "@neymar just signed a contract with a new club",
]

print("\nAugmented spaCy + EntityRuler on Twitter Samples")
print("=" * 60)

for tweet in test_tweets_ruler:
    doc = nlp_twitter(tweet)
    print(f"\nTweet: {tweet}")
    if doc.ents:
        for ent in doc.ents:
            print(f"  → {ent.text:<30} {ent.label_}")
    else:
        print("  → No entities detected!")


Added 11 Twitter-specific patterns

Augmented spaCy + EntityRuler on Twitter Samples

Tweet: Just followed @BarackObama on Twitter, amazing content
  → @BarackObama                   PERSON
  → Twitter                        ORG

Tweet: Bought $TSLA at 200, now at 250 thanks to Elon
  → TSLA                           PERSON
  → 200                            CARDINAL
  → 250                            CARDINAL

Tweet: lakers vs warriors tonight 🏀 who's winning?
  → lakers                         ORG
  → warriors                       ORG
  → tonight                        TIME

Tweet: @neymar just signed a contract with a new club
  → @neymar                        PERSON


---
## Part 6: Export Data for Annotation

We export pretrained model predictions in two formats:
1. **CoNLL format** — for model training
2. **Label Studio JSON** — for human review and correction in annotation tool


In [ ]:
# ============================================================
# EXPORT spaCy PREDICTIONS AS CoNLL FORMAT
# ============================================================

def export_conll(texts, nlp_model, output_file):
    """
    Export NER predictions in CoNLL (IOB2) format.

    Each token gets a tag:
      B-LABEL → first token of an entity
      I-LABEL → continuation tokens of an entity
      O       → not an entity (Outside)

    Args:
        texts: list of raw text strings
        nlp_model: spaCy NLP model
        output_file: path to write CoNLL file
    """
    with open(output_file, "w", encoding="utf-8") as f:
        for text in texts:
            doc = nlp_model(text)
            for sent in doc.sents:
                for token in sent:
                    if token.ent_iob_ == "O":
                        tag = "O"
                    else:
                        tag = f"{token.ent_iob_}-{token.ent_type_}"
                    f.write(f"{token.text}\t{tag}\n")
                f.write("\n")  # blank line between sentences

    print(f"CoNLL file written to: {output_file}")

# Use first 50 tweets as export sample
sample_texts = [reconstruct_text(dataset['train'][i]['tokens']) for i in range(50)]
export_conll(sample_texts, nlp, "twitter_spacy_predictions.conll")

# Preview the file
print("\n--- CoNLL File Preview (first 30 lines) ---")
with open("twitter_spacy_predictions.conll") as f:
    lines = f.readlines()
    for line in lines[:30]:
        print(line, end="")


CoNLL file written to: twitter_spacy_predictions.conll

--- CoNLL File Preview (first 30 lines) ---
I	O
hate	O
the	O
words	O
chunder	O
,	O
vomit	O
and	O
puke	O
.	O

BUUH	O
.	O

♥	O
.	O
.	O
)	O
)	O

(	O
♫	O
.	O
(	O
ړײ	O
)	O
♫	O
.	O

♥	B-ORG


In [ ]:
# ============================================================
# EXPORT spaCy PREDICTIONS AS LABEL STUDIO JSON
# ============================================================

def export_label_studio_json(texts, nlp_model, output_file):
    """
    Export NER predictions in Label Studio JSON format for import.

    Creates pre-annotated tasks that can be imported into Label Studio
    for human review and correction.

    The key: predictions go in "predictions" field (not "annotations")
    so Label Studio shows them as editable suggestions.

    Args:
        texts: list of raw text strings
        nlp_model: spaCy NLP model
        output_file: path to write JSON file
    """
    tasks = []
    for text in texts:
        doc = nlp_model(text)

        results = []
        for ent in doc.ents:
            results.append({
                "from_name": "label",      # must match Label Studio config
                "to_name": "text",          # must match Label Studio config
                "type": "labels",
                "value": {
                    "start": ent.start_char,
                    "end": ent.end_char,
                    "text": ent.text,
                    "labels": [ent.label_],
                },
            })

        task = {
            "data": {"text": text},
            "predictions": [{"result": results}] if results else [],
        }
        tasks.append(task)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(tasks, f, indent=2)

    print(f"Label Studio JSON written to: {output_file} ({len(tasks)} tasks)")

export_label_studio_json(sample_texts, nlp, "twitter_label_studio.json")

# Preview first task
print("\n--- Label Studio JSON Preview (first task) ---")
with open("twitter_label_studio.json") as f:
    data = json.load(f)
    print(json.dumps(data[0], indent=2))


Label Studio JSON written to: twitter_label_studio.json (50 tasks)

--- Label Studio JSON Preview (first task) ---
{
  "data": {
    "text": "I hate the words chunder , vomit and puke . BUUH ."
  },
  "predictions": []
}


In [ ]:
# ============================================================
# CONVERT BETWEEN FORMATS: Label Studio JSON ↔ CoNLL / JSONL
# ============================================================

def label_studio_to_iob2(ls_data):
    """
    Convert Label Studio JSON export to IOB2 token-label pairs.

    Handles the character-to-token alignment:
    - Tokenize by whitespace (preserving character positions)
    - Map character-level entity spans to token-level IOB tags

    Args:
        ls_data: list of Label Studio task dicts (parsed JSON)

    Returns:
        List of sentences, each a list of (token, tag) tuples
    """
    all_sentences = []

    for task in ls_data:
        text = task["data"]["text"]

        # Get annotations (completed) or predictions (pre-annotated)
        annotations = task.get("annotations", task.get("predictions", []))
        if not annotations:
            continue

        results = annotations[0].get("result", [])
        entities = []
        for r in results:
            if r.get("type") == "labels":
                val = r["value"]
                entities.append((val["start"], val["end"], val["labels"][0]))

        entities.sort(key=lambda x: x[0])

        # Tokenize and track character positions
        tokens = []
        token_spans = []
        for match in re.finditer(r'\S+', text):
            tokens.append(match.group())
            token_spans.append((match.start(), match.end()))

        # Initialize all tags as "O"
        tags = ["O"] * len(tokens)

        # Assign IOB tags based on character spans
        for ent_start, ent_end, label in entities:
            entity_started = False
            for i, (tok_start, tok_end) in enumerate(token_spans):
                if tok_start >= ent_start and tok_start < ent_end:
                    tags[i] = f"B-{label}" if not entity_started else f"I-{label}"
                    entity_started = True

        all_sentences.append(list(zip(tokens, tags)))

    return all_sentences


# Convert our Label Studio export back to IOB2
with open("twitter_label_studio.json") as f:
    ls_data = json.load(f)

sentences_iob = label_studio_to_iob2(ls_data)

print("Label Studio JSON → IOB2 conversion:")
for i, sent in enumerate(sentences_iob[:3]):
    print(f"\nSentence {i+1}:")
    for token, tag in sent:
        marker = " ←" if tag != "O" else ""
        print(f"  {token:<25} {tag}{marker}")


def iob2_to_conll(sentences, output_file):
    """Write IOB2 sentences to CoNLL format file."""
    with open(output_file, "w") as f:
        for sent in sentences:
            for token, tag in sent:
                f.write(f"{token}\t{tag}\n")
            f.write("\n")
    print(f"\nCoNLL file written to: {output_file}")

iob2_to_conll(sentences_iob, "from_label_studio_twitter.conll")

Label Studio JSON → IOB2 conversion:

Sentence 1:
  ♥                         O
  .                         O
  .                         O
  )                         O
  )                         O
  (                         O
  ♫                         O
  .                         O
  (                         O
  ړײ                        O
  )                         O
  ♫                         O
  .                         O
  ♥                         B-ORG ←
  .                         O
  «                         O
  ▓                         O
  »                         O
  ♥                         O
  .                         O
  ♫                         O
  .                         O
  .                         O
  ╝                         B-PERSON ←
  ╚                         I-PERSON ←
  .                         O
  .                         O
  ♫                         B-ORG ←
  Happy                     I-ORG ←
  New                       I-ORG ←
  Year  

In [ ]:
# ============================================================
# LOAD AND VISUALIZE LABEL STUDIO ANNOTATIONS
# ============================================================

def load_and_visualize_label_studio(filepath, max_items=5):
    """
    Load a Label Studio JSON export and display entities using displaCy.

    Args:
        filepath: path to Label Studio JSON export file
        max_items: maximum number of tasks to visualize
    """
    with open(filepath) as f:
        tasks = json.load(f)

    count = 0
    for task in tasks:
        if count >= max_items:
            break
        text = task["data"]["text"]

        annotations = task.get("annotations", task.get("predictions", []))
        if not annotations:
            continue

        results = annotations[0].get("result", [])
        ents = []
        for r in results:
            if r.get("type") == "labels":
                val = r["value"]
                ents.append({
                    "start": val["start"],
                    "end": val["end"],
                    "label": val["labels"][0],
                })

        if ents:
            doc_data = {"text": text, "ents": ents, "title": None}
            displacy.render(doc_data, style="ent", manual=True, jupyter=True)
            count += 1

print("Visualizing Label Studio pre-annotations (tweets with detected entities):\n")
load_and_visualize_label_studio("twitter_label_studio.json")


Visualizing Label Studio pre-annotations (tweets with detected entities):



---
## Summary: Notebook 1

### Model Performance on Twitter (Qualitative)

| Tool | Twitter Strengths | Twitter Weaknesses |
|------|-------------------|-------------------|
| **spaCy** | Good on capitalized formal names | Misses lowercase, @mentions, hashtags |
| **NLTK** | Transparent pipeline | Very low recall on social media text |
| **HuggingFace BERT** | Best recall across models | Still misses slang, @mentions |
| **Regex Rules** | Perfect for @mentions, $cashtags | Can't handle ambiguous names |
| **EntityRuler** | Augments spaCy with Twitter rules | Needs manual maintenance |

### Key Findings for Annotation
1. **All pretrained models struggle with lowercase entity names** — very common on Twitter
2. **@mentions are reliably person/org entities** — regex can pre-annotate these
3. **The corpus has 3 entity types** (PER, ORG, LOC) — clear and manageable
4. **Code-switching and abbreviations** require human judgment

### Next Steps
- **Notebook 2**: Train custom NER models (CRF, Bi-LSTM) on the Twitter corpus
- **Notebook 3**: End-to-end pipeline using your Label Studio annotations
